# Data Augmention using Generative LLMs


<center><img src="https://media.licdn.com/dms/image/v2/D4E12AQGLk5R0lcfr8A/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1713883053110?e=1737590400&v=beta&t=0b0YHpIGPl8KxMzLfEhhwFbkcrT3ZLz2BDTaax-4HNM" alt="llama" style="width:60%"></a></center>

### Setting things up:

First, we need to set some things up. This usually starts by **creating a virtual environment**.

Open your preferred command line. Make sure **Python** is available. This means you should be able to enter the Python promp if you type ```Python```.

**Navigate/create a folder** where you want to run your system. 

Inside that folder you can run the following commands:

```
conda deactivate           # deactivate conda in case you have it active
python -m venv venv       # create a folder 'venv' that will contain your virtual environment
source venv/bin/activate   # activate your virtual environment; you should now see '(venv)' on the left of your terminal prompt
```


You are now inside your clean virtual enviroment: nothing except python should be available. 
**Let's start installing the necessary things**!

**Run the following commands** inside your terminal:

```
pip install llama-cpp-python
pip install openai
pip install sse_starlette
pip install starlette_context
pip install pydantic_settings
pip install "fastapi[all]"
pip install jupyter
```

We should everything we need to run LLMs locally...  We're almost there! 

#### What are we missing?


# An LLM!  
### (Where can we find them? Aren't they huge?)

# Quantization
#### What is it? 
[Read more here](https://medium.com/@techresearchspace/what-is-quantization-in-llm-01ba61968a51)


**Advantages:**
 - Lesser memory consumption: Lower bit width results in less memory for storage
 - Fast Inference: This is due to efficient computation due to its lower memory bandwidth requirements
 - Less energy consumption: Larger model need more data movement and storage resulting in more energy consumption. Hence a smaller model results in compartively lesser energy usage.
 - Smaller models: Can quantize to suit the need and deploy to device with samller hadware specifications.

**Disadvantages:**
 - Potential loss in accuracy due to squeezing of high precision weight to lower precision.

# Using the LLM

### We'll be using Quantized LLMs, in GGUF format
[Click here to know more about and how to find GGUF models](https://huggingface.co/docs/hub/en/gguf)

[QuantFactory](https://huggingface.co/QuantFactory) is a good place to start!

### Once you download your GGUF model, you should be able to run it (from within the virtual environment):

`
python -m llama_cpp.server --host 0.0.0.0 --model ./models/Meta-Llama-3.1-8B-Instruct.Q3_K_M.gguf --n_ctx 8192
`

This launches the **LLM as a server** that **can be used while it is running**.
<br>

In [1]:
from collections import defaultdict as dd

from openai import OpenAI

# Point to the server
client = OpenAI(base_url="http://localhost:8000/v1", api_key="cltl")

# client = OpenAI(base_url="http://130.37.53.128:9002/v1", api_key="cltl")

## 1) In chat mode
- Try to understand how the contents of `history` is used
- What are the different `"roles"`?
- How to stop the chat?

In [2]:
history = [
    {
        "role": "system",
        "content": "You an chat-based assistant. Be cooperative, and polite. Try to be concise.",
    },
]

while True:
    completion = client.chat.completions.create(
        model="local-model",
        messages=history,
        temperature=0.8,
        stream=True,
    )

    new_message = {"role": "assistant", "content": ""}

    for chunk in completion:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
            new_message["content"] += chunk.choices[0].delta.content

    history.append(new_message)
    print("\n")
    userinput = input("> ")
    if userinput.lower() in ["bye", "quit", "exit"]:
        print("\n BYE BYE!")
        break
    history.append({"role": "user", "content": userinput})
    print("\n")

How can I assist you today?



>  who are you




I'm a computer program designed to have conversations with users like you. My purpose is to provide information, answer questions, and help with tasks in a friendly and helpful way. I'm here to listen and assist! What would you like to talk about or ask me?



>  exit



 BYE BYE!


## 2) In single prompt mode
- Is `history` still used?
- What does temperature do? What should theoretically happen if we set it to 0?


In [2]:
def query_LLM(model_client, prompt, temp=0.6):
    history = [
        {"role": "system", "content": prompt},
    ]
    completion = model_client.chat.completions.create(
        model="local-model",  # this field is currently unused
        messages=history,
        temperature=temp,
        stream=False,
    )

    return completion.choices[0].message.content

In [4]:
prompt = "what is coffee"
print(query_LLM(client, prompt, temp=0.6))

Coffee!

Coffee is a popular beverage made from the roasted seeds of the coffee plant, known as coffee beans. It's one of the most widely consumed beverages in the world and has been a staple in many cultures for centuries.

Here's a brief overview:

**What are coffee beans?**

Coffee beans are actually the seeds of the coffee plant (Coffea arabica or Coffea canephora), which is a flowering evergreen shrub native to tropical regions of Africa. The seeds are harvested, processed, and roasted to bring out their unique flavor and aroma.

**How is coffee made?**

The process involves:

1. **Harvesting**: Coffee cherries (the fruit that contains the beans) are picked from the plant.
2. **Pulping**: The outer skin of the cherry is removed, leaving just the inner seeds.
3. **Fermentation**: The seeds are fermented to break down some of the sugars and flavor compounds.
4. **Drying**: The seeds are dried to a moisture level that makes them easier to store and transport.
5. **Roasting**: The gre

In [125]:
prompt = "What is the best drink in the world? Your answer should contain the name of a single drink and nothing more."
print(query_LLM(client, prompt, temp=1))

Coffee.


# Data Augmentation -- Machine Translation as a case study
1) We must know what we want
2) We must know if we can/how to ask it
3) We must learn how to process its output

## 1) Who can help me understand what you need for your assignment?

## 2) We must know if we can/how to ask it
- Am I using the right LLM for the task?
- Should we be using the chat or the single prompt method?

In [26]:
prompt = "Can you translate text?"
# prompt = "What languages were you trained with?"
print(query_LLM(client, prompt, temp=0.6))

I can translate text from one language to another. I support translations in many languages, including but not limited to:

* European languages: English, Spanish, French, German, Italian, Portuguese, Dutch, Russian, Chinese (Simplified and Traditional), Japanese, Korean
* Indian languages: Hindi, Bengali, Marathi, Gujarati, Punjabi, Tamil, Telugu, Malayalam, Kannada
* Middle Eastern languages: Arabic, Hebrew, Persian (Farsi)
* Asian languages: Thai, Vietnamese, Indonesian, Malaysian

To translate text, you can:

1. Type or paste the text into this chat window.
2. Specify the source language and target language.

For example, you can say "Translate 'Hello' from English to Spanish" or simply type "translate hello en es".

Keep in mind that my translation abilities are based on machine learning algorithms, so while I strive for accuracy, there may be occasional errors or nuances that get lost in translation. If you need a professional or high-stakes translation, it's always best to consu

## 3) We must learn how to process its output
- How do we stop all the yapping?
- What is the difference between zero-/one-/few-shot prompting?
- How do we get the ouput in the format we want?

In [3]:

prompt = """
Please provide 5 english sentences and their respective Hindi translations. Each English sentence should be between 10 and 15 words long and must contain an idiom.
Your answer should be a list of list in python. The first element of each list should contain the English sentence and the second element should contain the Hindi translation.
Provide only the list and nothing else. For example:
[["english sentence"], ["hindi translation], ...]
"""

mt_list = []

import json

while len(mt_list) < 5:
    answer = query_LLM(client, prompt, temp=0.2)

    try:
        a = json.loads(answer)

        print('SUCCESS')
        print(answer)
        print('\n')
        for item in a:
            mt_list.append(item)
    except:
        print("Failed to parse: ", answer)
        print("Trying again!\n\n\n")

print("\n\n\nDONE:")
for pair in mt_list:
    print(pair)
    print("\n")

Failed to parse:  [
["It's raining cats and dogs outside, so we can't go to the park today.", "बाहर बारिश है इसलिए हमें पार्क नहीं जा सकते"],
["The new employee was a fish out of water in the meeting with the CEO.", "नई कर्मचारी मीटिंग में सीईओ के साथ थी एक मछली"],
["He's burning the midnight oil to finish this project on time, but it's not working.", "वह समय पर इस प्रोजेक्ट को पूरा करने के लिए मध्य रात्रि को जला रहा है लेकिन यह काम नहीं कर रहा"],
["The company is in a tight spot and needs to make some tough decisions quickly.", "कंपनी में एक संकट है और जल्दी से कुछ कठिन निर्णय लेने की जरूरत है"],
["The new policy has been a thorn in the side of many employees, causing frustration.", "नई नीति कई कर्मचारियों के लिए एक दर्दनाक बिंदु बन गई है, जिससे असंतुष्टि हो रही है"]
Trying again!





KeyboardInterrupt: 

# Quality Estimation / "Evaluation" with BLEU

<center><img src="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEhdPDCCJH2WVAuEWUvp-RWdQITk9L8dB2p62GVI9CzLHd_hC2cED4wovkTY07sSZmYHtiWcHbSUhPRzbg_2DYyHiq_9gElMN85ZwZAI2gPcuwQNleQATdqUlrd8klzjOLhvh-weaAWdqkA2/s1600/BLEU4.png" alt="llama" style="width:90%"></a><br><a href="https://kv-emptypages.blogspot.com/2019/04/understanding-mt-quality-bleu-scores.html">Taken from/Read more here</a></center>



In [ ]:
# !pip install nltk

In [6]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.util import ngrams

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\NIKHIL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
source_sent = "le professeur est arrivé en retard à cause de la circulation"
reference_transl = "the teacher arrived late because of the traffic"
reference_transl_tok = nltk.word_tokenize(reference_transl)
ngrams_reference = []
for n in [1, 2, 3, 4]:
    ngrams_reference = ngrams_reference + list(ngrams(reference_transl_tok, n))

print(ngrams_reference)

[('the',), ('teacher',), ('arrived',), ('late',), ('because',), ('of',), ('the',), ('traffic',), ('the', 'teacher'), ('teacher', 'arrived'), ('arrived', 'late'), ('late', 'because'), ('because', 'of'), ('of', 'the'), ('the', 'traffic'), ('the', 'teacher', 'arrived'), ('teacher', 'arrived', 'late'), ('arrived', 'late', 'because'), ('late', 'because', 'of'), ('because', 'of', 'the'), ('of', 'the', 'traffic'), ('the', 'teacher', 'arrived', 'late'), ('teacher', 'arrived', 'late', 'because'), ('arrived', 'late', 'because', 'of'), ('late', 'because', 'of', 'the'), ('because', 'of', 'the', 'traffic')]


In [8]:
transl_list = [
    "The professor was delayed due to the congestion",
    "Congestion was responsible for the teacher being late",
    "The teacher was late due to the traffic",
    "The professor arrived late because circulation",
    "The teacher arrived late because of the traffic",
]

transl_list_tokenized = []
for sent in transl_list:
    transl_list_tokenized.append(nltk.word_tokenize(sent))

print(transl_list_tokenized)

[['The', 'professor', 'was', 'delayed', 'due', 'to', 'the', 'congestion'], ['Congestion', 'was', 'responsible', 'for', 'the', 'teacher', 'being', 'late'], ['The', 'teacher', 'was', 'late', 'due', 'to', 'the', 'traffic'], ['The', 'professor', 'arrived', 'late', 'because', 'circulation'], ['The', 'teacher', 'arrived', 'late', 'because', 'of', 'the', 'traffic']]


In [9]:
from nltk.translate.bleu_score import sentence_bleu

ngram_weights = (0.10, 0.30, 0.30, 0.30)  # weights for 1-gram, 2-gram, 3-gram, 4-gram

for translation in transl_list_tokenized:

    # Fine the translation n-grams
    # Not needed for the score, just to see the overlap
    sent_ngrams = []
    for n in [1, 2, 3, 4]:
        sent_ngrams = sent_ngrams + list(ngrams(translation, n))

    bleu_score1 = sentence_bleu(
        [reference_transl_tok], translation
    )  # This can be a list of references
    bleu_score2 = sentence_bleu(
        [reference_transl_tok], translation, weights=ngram_weights
    )  # This can be a list of references

    print(bleu_score1, bleu_score2)

    # 1-gram overlap
    print(set(ngrams_reference) & set(sent_ngrams))
    print("\n\n")

1.0832677820940877e-231 1.052691193011681e-277
{('the',)}



7.176381577237209e-155 1.2950316234712509e-185
{('late',), ('the',), ('teacher',), ('the', 'teacher')}



7.711523862191631e-155 1.3328284280434942e-185
{('the', 'traffic'), ('the',), ('late',), ('traffic',), ('teacher',)}



4.1382219658909647e-78 1.695647221393335e-93
{('late', 'because'), ('arrived', 'late'), ('because',), ('late',), ('arrived', 'late', 'because'), ('arrived',)}



0.8408964152537145 0.834236890454548
{('because', 'of', 'the'), ('arrived', 'late'), ('late',), ('because', 'of', 'the', 'traffic'), ('teacher',), ('arrived',), ('teacher', 'arrived', 'late'), ('of', 'the'), ('arrived', 'late', 'because'), ('traffic',), ('teacher', 'arrived'), ('late', 'because', 'of'), ('late', 'because', 'of', 'the'), ('the',), ('because',), ('of', 'the', 'traffic'), ('because', 'of'), ('late', 'because'), ('the', 'traffic'), ('arrived', 'late', 'because', 'of'), ('teacher', 'arrived', 'late', 'because'), ('of',)}





E:\Vrije_University\Courses\Language_as_Data\Llama\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
E:\Vrije_University\Courses\Language_as_Data\Llama\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
E:\Vrije_University\Courses\Language_as_Data\Llama\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of l

## Why is the last one not 1.0?